# P1. Очистка данных

Вход: `data/merged.csv` (Фонтанка, Интерфакс, Лента за 2016, колонки `source, date, time, title, text`).

Шаги:
1. Нормализация: NFKC, кавычки, тире, пробелы, дата/время в UTC.
2. Удаление служебных блоков ("Подписывайтесь...", "Читайте также" и т.п.), повторов внутри текста, подписей изданий в конце.
3. Фильтры: язык ru, длина текста > 200, доля букв > 0.6, есть заголовок и дата.

Дубликаты между статьями не удаляются, это P3.

Выход: `data/clean/news_2016.csv` (`source, dt_utc, title, text`), отсеянные в `data/rejected/` с колонкой `reason`.

In [ ]:
import html
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

DATA = Path.cwd().parent / "data"
SRC = DATA / "merged.csv"
CLEAN = DATA / "clean"
REJECTED = DATA / "rejected"
REPORTS = DATA / "reports"

TZ = "Europe/Moscow"
MIN_TEXT_LEN = 200
MIN_LETTER_SHARE = 0.6
MIN_CYR_SHARE = 0.85  # доля кириллицы среди букв, проверка на ru
BOILERPLATE_SAMPLE = 20_000  # размер выборки для поиска шаблонных блоков
BOILERPLATE_MIN_DOCS = 20  # мин. документов выборки с блоком

COLUMNS = ["source", "dt_utc", "title", "text"]

## Нормализация

In [ ]:
# нестандартные пробелы
SPACES = re.compile("[\u00a0\u1680\u2000-\u200a\u202f\u205f\u3000]")

# управляющие и невидимые символы (zero-width, soft hyphen, BOM)
INVISIBLE = re.compile("[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\u00ad\u200b-\u200f\u2060\ufeff]")

# двойные кавычки -> «», одинарные -> '
QUOTES = str.maketrans({
    "“": "«", "”": "»", "„": "«", "‟": "«", "″": "«", "‹": "«", "›": "»", "ʺ": "«",
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "′": "'", "ʹ": "'", "`": "'", "´": "'",
})
# прямые парные кавычки -> «» (у Интерфакса прямые)
PAIRED_QUOTES = re.compile(r'"([^"]{1,300}?)"')

# тире и минусы -> дефис, отдельно стоящий дефис -> тире
DASHES = re.compile("[\u2010-\u2015\u2043\u2212\ufe58\ufe63\uff0d]")

TAGS = re.compile(r"<[^>]{1,200}>")
URLS = re.compile(r"\(?\b(?:https?://|www\.)\S+\)?", re.I)

# NFKC превращает № в No, временно заменяем
NUMERO = "\x01"

def normalize(s):
    s = html.unescape(s)
    s = unicodedata.normalize("NFKC", s.replace("№", NUMERO)).replace(NUMERO, "№")
    s = TAGS.sub(" ", s)
    s = URLS.sub(" ", s)
    s = INVISIBLE.sub("", s)
    s = SPACES.sub(" ", s)
    s = s.translate(QUOTES)
    s = PAIRED_QUOTES.sub("«\\1»", s)
    s = DASHES.sub("-", s)
    s = re.sub(r"(?<=\s)-{1,3}(?=\s)", "—", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s+([,.;:!?%])", r"\1", s)  # пробел перед знаком препинания
    return s.strip()

## Служебные блоки

In [ ]:
# маркеры служебных предложений, такие предложения удаляются целиком
JUNK = re.compile(
    r"подписывайтесь|подписка на"
    r"|читайте\s+(также|далее|подробнее|по\s+теме|нас\s+в)"
    r"|смотрите\s+(также|видео)"
    r"|(все\s+)?материалы\s+(по\s+теме|сюжета)"
    r"|следите\s+за\s+(нами|новостями)"
    r"|наш\w*\s+(канал|телеграм|telegram|паблик)"
    r"|нашли\s+ошибку|ctrl\s*\+\s*enter"
    r"|все\s+права\s+защищены|на\s+правах\s+рекламы"
    r"|обновите\s+браузер|flash\s*-?\s*плагин|необходимо\s+обновиться"
    r"|версия\s+для\s+печати|поделиться\s+в\s+соцсетях"
    r"|^(фото|видео|инфографика|источник)\s*:",
    re.I,
)

# подпись издания в конце текста
TAIL = re.compile(r"фонтанка|интерфакс|лента\.ру|lenta\.ru|interfax|\bИА\b|корреспондент\s+\w+$", re.I)

# граница предложений, без сокращений (г., ул., инициалы)
SENT = re.compile(
    r"(?<=[.!?…])"
    r"(?<!\bг\.)(?<!\bул\.)(?<!\bд\.)(?<!\bтыс\.)(?<!\bмлн\.)(?<!\bмлрд\.)(?<!\s[А-ЯЁ]\.)"
    r"\s+(?=[«(\[]?[А-ЯЁA-Z0-9])"
)

def sentences(s):
    return [p.strip() for p in SENT.split(s) if p.strip()]

# translate быстрее regex
PUNCT = str.maketrans({c: " " for c in "«»\"'`.,;:!?()[]{}—–-…%№*/\\|+=<>@#$^&~"})


def key(sent):
    # нормализованное предложение для сравнения
    return " ".join(sent.lower().translate(PUNCT).split())

def drop_junk(text, boilerplate):
    # JUNK по предложениям проверяем только если он есть в тексте
    has_junk = JUNK.search(text) is not None
    out, seen = [], set()
    for s in sentences(text):
        k = key(s)
        if not k or k in boilerplate or k in seen:
            continue
        if has_junk and JUNK.search(s):
            continue
        seen.add(k)
        out.append(s)
    # короткое последнее предложение с названием издания - подпись
    if out and len(out[-1]) <= 80 and TAIL.search(out[-1]):
        out.pop()
    return " ".join(out)

## Очистка

In [ ]:
def clean_news():
    df = pd.read_csv(SRC, dtype="string")
    total = len(df)
    print(f"прочитано: {total}")

    # дата и время -> UTC
    naive = pd.to_datetime(df["date"] + " " + df["time"].fillna("00:00:00"), errors="coerce")
    df["dt_utc"] = naive.dt.tz_localize(TZ, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")

    # нормализация
    df["title"] = df["title"].fillna("").map(normalize)
    df["text"] = df["text"].fillna("").progress_map(normalize)

    # шаблонные предложения (дисклеймеры, врезки), частота по выборке, длиной до 400
    sample = df["text"].sample(min(BOILERPLATE_SAMPLE, len(df)), random_state=0)
    counter = Counter()
    for text in sample:
        counter.update({key(s) for s in sentences(text) if 20 < len(s) <= 400})
    boilerplate = {k for k, n in counter.items() if n >= BOILERPLATE_MIN_DOCS}
    print(f"шаблонных блоков найдено: {len(boilerplate)}")

    df["text"] = df["text"].progress_map(lambda t: drop_junk(t, boilerplate))

    # метрики; буквы явно, т.к. со string dtype регулярки через pyarrow (RE2), там \w только латиница
    length = df["text"].str.len()
    letters = df["text"].str.count(r"[A-Za-zА-Яа-яЁё]")
    cyr = df["text"].str.count(r"[А-Яа-яЁё]")
    lat = df["text"].str.count(r"[A-Za-z]")

    # фильтры
    checks = pd.DataFrame({
        "no_date": df["dt_utc"].isna(),
        "no_title": df["title"].str.len() < 10,
        "short_text": length <= MIN_TEXT_LEN,
        "low_letter_share": letters / length.where(length > 0) <= MIN_LETTER_SHARE,
        "not_ru": cyr / (cyr + lat).where(cyr + lat > 0) < MIN_CYR_SHARE,
    }).fillna(True)

    keep = ~checks.any(axis=1)
    # причины отсева через ;
    df["reason"] = [";".join(checks.columns[row]) for row in checks.to_numpy()]

    df = df.sort_values("dt_utc", na_position="last")
    keep = keep.reindex(df.index)

    clean = df.loc[keep, COLUMNS].reset_index(drop=True)
    rejected = df.loc[~keep, COLUMNS + ["reason"]].reset_index(drop=True)

    # сохранение
    for folder in (CLEAN, REJECTED, REPORTS):
        folder.mkdir(parents=True, exist_ok=True)
    clean.to_csv(CLEAN / "news_2016.csv", index=False)
    rejected.to_csv(REJECTED / "news_2016.csv", index=False)

    # статистика
    stats = checks.sum().sort_values(ascending=False).rename("записей").to_frame()
    stats["доля, %"] = (stats["записей"] / total * 100).round(2)
    stats.index.name = "проверка"
    stats.to_csv(REPORTS / "p1_reject_stats.csv")

    print(stats.to_string())
    print(f"\nчисто:   {len(clean)} ({len(clean) / total:.1%}) -> {CLEAN}")
    print(f"отсеяно: {len(rejected)} -> {REJECTED}")
    return clean, rejected, stats

In [ ]:
clean, rejected, stats = clean_news()
clean.head()

## Артефакты для отчёта

Источники, распределение по датам, длины текстов.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

clean["source"].value_counts().plot.bar(ax=axes[0], title="Статей по источникам", rot=0)

by_day = clean["dt_utc"].dt.tz_convert(TZ).dt.date.value_counts().sort_index()
by_day.plot(ax=axes[1], title="Статей по датам")

lengths = clean["text"].str.len()
lengths.plot.hist(ax=axes[2], bins=60, range=(0, 8000), title="Длина текста, символов")

plt.tight_layout()
plt.show()

print(f"в корпусе {len(clean)} статей, "
      f"{clean['dt_utc'].min():%Y-%m-%d} — {clean['dt_utc'].max():%Y-%m-%d}")
print(lengths.describe().round(0).to_string())